# 04 Modelo Regresion Total Pedido

Objetivo: estimar el valor de `total_pedido` a partir de la base por ticket.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "base_tickets_modelado.parquet"
df = pd.read_parquet(PARQUET_PATH)
df.head()

In [ ]:
target = "total_pedido"
columnas_excluir = ["id_ticket_modelado", "fecha", "total_pedido", "monto_pago"]
X = df.drop(columns=columnas_excluir)
y = df[target]
X.shape, y.shape

In [ ]:
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numericas
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categoricas
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Modelo base sugerido
modelo = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Alternativa sugerida: RandomForestRegressor(random_state=42)
modelo